# Visual Health — confirmation sur DDR (Kaggle)

Embeddings RETFound (224 px) et RETFound-Green (392 px), prétraitement `crop-mask`,
des 12 522 images gradables de **DDR** (rétinopathie diabétique, Chine, CC BY-NC-SA 4.0).
Protocole fixé avant tout calcul : `scripts/confirmation_ddr.py` (commit `fb480ea`).
L'analyse se fait ensuite en local, sur CPU.

Le notebook vérifie d'abord que le dataset envoyé est **identique à l'archive des
auteurs** (empreinte de chaque image) et s'arrête sinon.

**Avant de lancer** :
1. `Add Input` → onglet **Your Datasets** → le dataset privé DDR (zip `ddr_dr_grading.zip`).
2. `Settings > Accelerator > GPU T4 x2` (pas P100), `Settings > Internet > On`.
3. `Add-ons > Secrets` : `HF_TOKEN` coché (poids RETFound).
4. **`Save Version > Save & Run All (Commit)`** (~30 à 45 min). Le fichier
   `visual_health_ddr_embeddings.zip` apparaît ensuite dans l'onglet `Output`.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : activer GPU T4 x2"
!df -h /kaggle/working /kaggle/temp | tail -2


In [ ]:
import os
import subprocess

OUTPUT = "/kaggle/working"  # seul le zip y est écrit
WORK = "/kaggle/temp"
os.makedirs(WORK, exist_ok=True)
%cd {WORK}

REPO = "sumarc183-design/visual-health-platform"
REVISION = "main"  # ou un commit précis, pour rejouer un run à l'identique


def _git(*args):
    if subprocess.run(["git", *args], capture_output=True).returncode != 0:
        raise RuntimeError(f"git {args[0]} a échoué")


if not os.path.exists("visual-health-platform"):
    _git("clone", "-q", f"https://github.com/{REPO}.git", "visual-health-platform")
_git("-C", "visual-health-platform", "fetch", "-q", "origin")
_git("-C", "visual-health-platform", "reset", "-q", "--hard", "origin/main" if REVISION == "main" else REVISION)
!cd visual-health-platform && git log --oneline -1
%cd {WORK}/visual-health-platform
!pip install -q -r requirements.txt


## Vérification du dataset

Empreinte de l'inventaire officiel (fichiers de stades et 13 673 images), calculée sur
l'archive des auteurs : `inventory_digest` dans `src/data/ddr_dataset.py`.


In [ ]:
import glob

hits = sorted(glob.glob("/kaggle/input/**/DR_grading/train.txt", recursive=True))
if not hits:
    raise RuntimeError(f"DDR introuvable : ajouter le dataset privé en input. Inputs : {os.listdir('/kaggle/input')}")
DDR = os.path.dirname(os.path.dirname(hits[0]))
print("DDR :", DDR)

from src.data.ddr_dataset import inventory_digest

EXPECTED = "b05f6d1711402a1e1a77f5dd0e0a7f32741d6780dd604b18533c482d7611cfcd"
digest, n = inventory_digest(DDR)
print(n, "fichiers, empreinte", digest)
if digest != EXPECTED:
    raise RuntimeError("Le dataset DDR diffère de l'archive officielle : arrêt avant tout calcul")
print("Dataset identique à l'archive officielle")


In [ ]:
import glob

TARGET = "third_party/retfound/checkpoints/RETFound_mae_natureCFP.pth"
os.makedirs(os.path.dirname(TARGET), exist_ok=True)
found = sorted(glob.glob("/kaggle/input/**/RETFound_mae_natureCFP.pth", recursive=True))
if os.path.exists(TARGET):
    print("Poids RETFound déjà présents")
elif found:
    os.symlink(found[0], TARGET)
    print("Poids RETFound pris dans les inputs :", found[0])
else:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")  # jamais écrit dans le notebook
    !python scripts/download_retfound_checkpoint.py --repo-id YukunZhou/RETFound_mae_natureCFP --out third_party/retfound/checkpoints
CHECKPOINT = TARGET

GREEN = "third_party/retfound_green/retfoundgreen_statedict.pth"
if not os.path.exists(GREEN):
    !mkdir -p third_party/retfound_green
    !curl -sL --fail -o {GREEN} https://github.com/justinengelmann/RETFound_Green/releases/download/v0.1/retfoundgreen_statedict.pth
GREEN_SHA = "431de5dbc1bebbb32f60e2c0bcf8daa4f8bcbf06f7cb1e1dc97ec589713942e1"
found = !sha256sum {GREEN}
if found[0].split()[0] != GREEN_SHA:
    raise RuntimeError(f"Poids RETFound-Green inattendus : {found[0]}")
print(CHECKPOINT, GREEN, "OK")


## Embeddings

Même extraction que pour APTOS et EyePACS (`scripts/extract_embeddings.py`), cache avec
empreinte du checkpoint et du prétraitement ; `ddr_labels.csv` : clé, stade, groupe, format.


In [ ]:
import time

OUTS = {
    "retfound": "embeddings/ddr_crop-mask_224_RETFound_mae_natureCFP.npz",
    "green": "embeddings/ddr_crop-mask_392_retfoundgreen_statedict.npz",
}
for backbone, ckpt in (("retfound", CHECKPOINT), ("green", GREEN)):
    t0 = time.time()
    !python -u scripts/extract_embeddings.py --dataset ddr --backbone {backbone} --data-root "{DDR}" \
        --checkpoint {ckpt} --out-dir embeddings --batch-size 64 --num-workers 4 2>&1 | grep -v -i warning
    out = OUTS[backbone]
    if not os.path.exists(out) or os.path.getmtime(out) < t0:
        raise RuntimeError(f"extract_embeddings.py a échoué pour {backbone} (voir la sortie ci-dessus)")
    print(backbone, f"{(time.time() - t0) / 60:.0f} min")
if not os.path.exists("embeddings/ddr_labels.csv"):
    raise RuntimeError("ddr_labels.csv absent")


## Garder le résultat


In [ ]:
import zipfile

archive = f"{OUTPUT}/visual_health_ddr_embeddings.zip"
with zipfile.ZipFile(archive, "w") as z:
    for f in (*OUTS.values(), "embeddings/ddr_labels.csv"):
        z.write(f)
subprocess.run(["python", "scripts/run_manifest.py", "--archive", archive, "--requested-revision", REVISION], check=True)
print("->", archive, f"({os.path.getsize(archive) / 1e6:.0f} Mo)")
